In [ ]:
!pip install segmentation-models-pytorch

In [ ]:
import os
import random
from glob import glob
from pathlib import Path
import segmentation_models_pytorch as smp

import matplotlib.pyplot as plt

import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torch.utils.data import Dataset

from tqdm.auto import tqdm

import pydicom
from pydicom.pixels.processing import convert_color_space

import numpy as np
from PIL import Image

In [ ]:
import os

IN_KAGGLE = False
if os.environ.get("KAGGLE_URL_BASE", ""):
    IN_KAGGLE = True
print(IN_KAGGLE)

In [ ]:
from pathlib import Path
import shutil

src_dir = Path("/kaggle/input/notebooks/riccardogibello11/segmentation")
dst_dir = Path.cwd()  # Automatically resolves to /kaggle/working

# List patterns to ignore (Notebooks, JSON, Images, Reports, outputs, etc.)
ignored_extensions = shutil.ignore_patterns(
    "*.ipynb",
    "*.json",
    "*.png",
    "*.jpg",
    "*.jpeg",
    "*.html",
    "*.csv",
    "*.parquet",
    "*.txt",
    ".ipynb_checkpoints",
    "__pycache__",
)

if src_dir.exists():
    print(f"Copying Python files from {src_dir} to {dst_dir}...")
    shutil.copytree(src_dir, dst_dir, ignore=ignored_extensions, dirs_exist_ok=True)
    print("Copy completed successfully!")
else:
    print(f"Source path {src_dir} does not exist.")

In [ ]:
# Local runs: this folder must contain the `camus` directory (change it if needed)
LOCAL_ROOT = Path.home() / "Downloads" / "release"

if IN_KAGGLE:
    DATA_ROOT = Path("/kaggle/input/datasets/riccardogibello11/camus-example/camus")
    OUTPUT_ROOT = Path("/kaggle/working")
else:
    DATA_ROOT = LOCAL_ROOT / "camus"
    OUTPUT_ROOT = LOCAL_ROOT
if not DATA_ROOT.exists():
    raise FileNotFoundError(f"{DATA_ROOT} not found: please update LOCAL_ROOT")

# Embeddings and model weights computed by this notebook are stored here
MODELS_ROOT = OUTPUT_ROOT / "models"
MODELS_ROOT.mkdir(parents=True, exist_ok=True)

In [ ]:
RUN_SMALL_DATASET = False
# Fixed seed: re-running the notebook gives the same data order,
# the same random augmentations and the same starting weights
SEED = 42
# False: every run trains from scratch. True: continue from the latest saved checkpoint
RESUME_TRAINING = False

MODELS_DIR_PATH = OUTPUT_ROOT / "models"
MODEL_NAME = "echo-resnet18"
MODEL_FOLDER_PATH = MODELS_DIR_PATH / MODEL_NAME
MODEL_CHECKPOINT_FOLDER_PATH = MODEL_FOLDER_PATH / "checkpoints"
os.makedirs(MODEL_CHECKPOINT_FOLDER_PATH, exist_ok=True)

In [ ]:
def load_dataset_map(
    data_dir,
):
    # Use the glob function to find all the ".dcm"
    # files in the data directory and its subdirectories,
    # excluding the "perturbed" and "masks" subdirectories.
    # Sorted, so the file order (and the small subset) is the same on every machine
    dicom_files = sorted(glob(os.path.join(data_dir, "**", "*.dcm"), recursive=True))
    dicom_files = [
        f
        for f in dicom_files
        if not any([value in f for value in ["perturbed", "masks"]])
    ]
    # Create a dictionary containing the three splits (train, val, test),
    # each containing the "mask" and "img" keys with the corresponding file paths.
    dataset_dict = {
        "train": [
            {"img": f, "mask": f.replace("clean", "masks")}
            for f in dicom_files
            if "train" in f
        ],
        "val": [
            {"img": f, "mask": f.replace("clean", "masks")}
            for f in dicom_files
            if "val" in f
        ],
        "test": [
            {"img": f, "mask": f.replace("clean", "masks")}
            for f in dicom_files
            if "test" in f
        ],
    }
    return dataset_dict

In [ ]:
def augment_image_and_mask(
    img: torch.Tensor, mask: torch.Tensor
) -> tuple[torch.Tensor, torch.Tensor]:
    """Randomly flip and crop an image (1, H, W) and its mask (1, H, W) together."""

    def coin() -> bool:
        return torch.rand(1).item() < 0.5

    # Only left-right flips: upside-down flips and 90° rotations would put the tip
    # of the ultrasound fan at the bottom or on the side, which never happens in CAMUS
    if coin():  # horizontal flip
        img, mask = img.flip(2), mask.flip(2)
    if coin():  # crop, centred on the LV half of the time
        img, mask = random_crop(img, mask, center_on_mask=coin())

    return img, mask


def random_crop(
    img: torch.Tensor,
    mask: torch.Tensor,
    center_on_mask: bool,
    scale_range: tuple[float, float] = (0.7, 0.9),
) -> tuple[torch.Tensor, torch.Tensor]:
    """Crop 70-90% of the image (centred on the LV or random), then resize back to (H, W)."""
    _, H, W = img.shape
    scale = torch.empty(1).uniform_(*scale_range).item()
    ch, cw = max(1, int(H * scale)), max(1, int(W * scale))

    fg = torch.nonzero(mask[0] == 1)  # LV pixels (label 1)
    if center_on_mask and len(fg) > 0:
        cy, cx = ((fg.min(0).values + fg.max(0).values) // 2).tolist()
        y = min(max(0, cy - ch // 2), H - ch)
        x = min(max(0, cx - cw // 2), W - cw)
    else:
        y = torch.randint(0, H - ch + 1, (1,)).item()
        x = torch.randint(0, W - cw + 1, (1,)).item()

    img = img[:, y : y + ch, x : x + cw]
    mask = mask[:, y : y + ch, x : x + cw]

    # Resize back so every sample in a batch has the same shape.
    # "nearest-exact" copies label values without blending them and stays aligned
    # with the bilinear image resize ("nearest" shifts the mask by about half a pixel)
    img = F.interpolate(img[None], size=(H, W), mode="bilinear", align_corners=False)[0]
    mask = F.interpolate(mask[None].float(), size=(H, W), mode="nearest-exact")[0].to(
        mask.dtype
    )
    return img, mask

In [ ]:
def read_dicom_image(file_path: str, frame: int = 0, size: int = 224) -> torch.Tensor:
    """Read one frame of a (possibly multi-frame, colour) DICOM as a (1, size, size) float tensor in [0, 1]."""
    ds = pydicom.dcmread(file_path)
    arr = ds.pixel_array

    if getattr(ds, "NumberOfFrames", 1) > 1:  # cine loop → pick one frame
        arr = arr[frame]

    if arr.ndim == 3:  # colour → grayscale
        if ds.PhotometricInterpretation.startswith("YBR"):
            arr = convert_color_space(arr, ds.PhotometricInterpretation, "RGB")
        arr = arr.astype(np.float32) @ np.array([0.299, 0.587, 0.114], dtype=np.float32)

    arr = arr.astype(np.float32)
    arr = (arr - arr.min()) / (arr.max() - arr.min() + 1e-8)  # bit-depth independent

    arr = np.array(Image.fromarray(arr).resize((size, size), Image.Resampling.BILINEAR))
    return torch.from_numpy(arr).unsqueeze(0)


def read_dicom_mask(file_path: str, size: int = 224) -> torch.Tensor:
    """Read a label mask as a (1, size, size) integer tensor, keeping the label values."""
    arr = pydicom.dcmread(file_path).pixel_array.astype(np.uint8)
    arr = np.array(Image.fromarray(arr).resize((size, size), Image.Resampling.NEAREST))
    return torch.from_numpy(arr).long().unsqueeze(0)


class CustomDataset(Dataset):
    def __init__(
        self,
        paths: list[dict[str, str]],
        augment: bool = False,
    ):
        self.imgs = [p["img"] for p in paths]
        self.masks = [p["mask"] for p in paths]
        # Keep only 5 samples per set if RUN_SMALL_DATASET is True
        if RUN_SMALL_DATASET:
            self.imgs = self.imgs[:5]
            self.masks = self.masks[:5]
        self.augment = augment

    def __len__(self):
        return len(self.imgs)

    def __getitem__(self, index) -> tuple[torch.Tensor, torch.Tensor]:
        img_path = self.imgs[index]
        if self.augment:
            img, mask = augment_image_and_mask(
                read_dicom_image(img_path), read_dicom_mask(self.masks[index])
            )
        else:
            img = read_dicom_image(img_path)
            mask = read_dicom_mask(self.masks[index])
        # Keep only the label set to 1, and set all other labels to 0,
        # to keep track only of the region of interest (ROI) in the mask
        # (i.e., the LV).
        mask = (mask == 1).float()
        # Read both DICOM files and return the images
        return img, mask


def collate_fn(
    batch: list[tuple[torch.Tensor, torch.Tensor]],
) -> tuple[torch.Tensor, torch.Tensor]:
    """
    Collate function to combine a list of samples into a batch.

    Args:
        batch (list): A list of tuples, where each tuple contains an image tensor and a mask tensor.

    Returns:
        tuple: A tuple containing two tensors: the batched images and the batched masks.
    """
    imgs, masks = zip(*batch)
    return torch.stack(imgs), torch.stack(masks)

In [ ]:
def dice_score(pred_mask: torch.Tensor, true_mask: torch.Tensor) -> torch.Tensor:
    """
    Compute the Dice score, which measures how much two masks overlap:
    2 * (overlap area) / (area of the first mask + area of the second mask).
    It is 1 when the two masks are identical and 0 when they do not overlap at all.

    Args:
        pred_mask (torch.Tensor): Predicted mask (probabilities or 0/1 values).
        true_mask (torch.Tensor): Ground truth mask (0/1 values).

    Returns:
        torch.Tensor: The Dice score.
    """
    eps = 1e-5  # avoids a division by zero (and gives 1 when both masks are empty)
    intersection = torch.sum(pred_mask * true_mask)
    total = torch.sum(pred_mask) + torch.sum(true_mask)
    return (2.0 * intersection + eps) / (total + eps)


class CustomDiceLoss(torch.nn.Module):
    """
    Soft Dice loss: 1 - Dice, with the Dice computed directly on the predicted
    probabilities (0-1 values), without first turning them into a 0/1 mask.

    Thresholding would make the loss useless for training: a small change of a weight
    almost never flips a pixel from 0 to 1, so the loss would not change and every
    gradient would be zero (the model would get no hint on how to improve).
    With probabilities, every small change of a weight changes the loss a little.
    All the pixels of the batch are pooled together into a single Dice score.
    """

    def forward(
        self,
        pred_mask: torch.Tensor,
        true_mask: torch.Tensor,
    ) -> torch.Tensor:
        """
        Compute the soft Dice loss between predicted and true masks.

        Args:
            pred_mask (torch.Tensor): Predicted probabilities.
            true_mask (torch.Tensor): Ground truth mask.

        Returns:
            torch.Tensor: The computed Dice loss.
        """
        return 1.0 - dice_score(pred_mask, true_mask)


def compute_dice(
    pred_mask: torch.Tensor,
    true_mask: torch.Tensor,
    threshold: float = 0.5,
) -> float:
    """
    Compute the hard Dice metric of one image: the probabilities are first turned into
    a 0/1 mask (pixels above the threshold belong to the LV), i.e. the mask a user of
    the model would actually see, and the Dice score is computed on that mask.
    It is the same formula as the loss, so the two numbers can be compared directly,
    and it is the metric reported in the CAMUS papers (averaged over the images).

    Args:
        pred_mask (torch.Tensor): Predicted probabilities of one image.
        true_mask (torch.Tensor): Ground truth mask of the same image.
        threshold (float): Threshold to binarize the predicted mask.

    Returns:
        float: The computed Dice value.
    """
    pred_mask_bin = (pred_mask > threshold).float()
    return dice_score(pred_mask_bin, true_mask).item()

In [ ]:
def visualize_losses(
    save_folder_path: str,
    train_losses: list[float],
    val_losses: list[float] | None = None,
):
    """
    Visualize the training and validation losses over epochs.

    Args:
        train_losses (list[float]): List of training losses.
        val_losses (list[float] | None): List of validation losses.
        save_folder_path (str): Path to the folder where the loss plot image will be saved.
    """
    os.makedirs(save_folder_path, exist_ok=True)
    epochs = range(1, len(train_losses) + 1)

    n_col = 1 if val_losses is None else 2
    _, ax = plt.subplots(1, n_col, figsize=(10, 5))
    ax.plot(epochs, train_losses, label="Training Loss")
    if val_losses is not None:
        ax.plot(epochs, val_losses, label="Validation Loss")
    ax.set_xlabel("Epochs")
    ax.set_ylabel("Loss")
    ax.set_title("Training and Validation Losses")
    ax.legend()
    ax.grid()
    plt.savefig(os.path.join(save_folder_path, "loss_plot.png"))
    plt.close()


def save_checkpoints(
    model: torch.nn.Module,
    optimizer: torch.optim.Optimizer,
    epoch: int,
    checkpoint_dir: str = "checkpoints",
    is_best: bool = False,
):
    """
    Save the model and optimizer state dictionaries as checkpoints.

    Args:
        model (torch.nn.Module): The model to save.
        optimizer (torch.optim.Optimizer): The optimizer to save.
        epoch (int): The current epoch number.
        checkpoint_dir (str): Directory to save the checkpoints.
        is_best (bool): If True, save the checkpoint as the best model.
    """
    os.makedirs(checkpoint_dir, exist_ok=True)
    if is_best:
        checkpoint_path = os.path.join(checkpoint_dir, f"best_checkpoint.pth")
    else:
        checkpoint_path = os.path.join(checkpoint_dir, f"checkpoint_epoch_{epoch}.pth")
    torch.save(
        {
            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
        },
        checkpoint_path,
    )
    # Delete any previous checkpoint files for the same epoch to save disk space
    if not is_best:
        previous_checkpoints = glob(
            os.path.join(checkpoint_dir, f"checkpoint_epoch_{epoch}_*.pth")
        )
        for prev_checkpoint in previous_checkpoints:
            if prev_checkpoint != checkpoint_path:
                os.remove(prev_checkpoint)


def load_checkpoint(
    checkpoint_dir: str,
    model: torch.nn.Module,
    optimizer: torch.optim.Optimizer,
    epoch: int | None = None,
    is_best: bool | None = None,
) -> int | None:
    """
    Load the model and optimizer state dictionaries from a checkpoint.

    Args:
        checkpoint_dir (str): Directory containing the checkpoints.
        model (torch.nn.Module): The model to load.
        optimizer (torch.optim.Optimizer): The optimizer to load.
        epoch (int | None): The epoch number of the checkpoint to load.
        is_best (bool | None): If True, load the best model checkpoint.
    """
    if is_best:
        checkpoint_path = os.path.join(checkpoint_dir, f"best_checkpoint.pth")
    elif epoch is not None:
        checkpoint_path = os.path.join(checkpoint_dir, f"checkpoint_epoch_{epoch}.pth")
    else:
        # Look into the checkpoint directory to possibly find the latest checkpoint file
        checkpoint_files = glob(os.path.join(checkpoint_dir, "checkpoint_epoch_*.pth"))
        if not checkpoint_files:
            return None  # No checkpoint files found
        # Sort the checkpoint files by epoch number and get the latest one
        checkpoint_files.sort(
            key=lambda x: int(os.path.basename(x).split("_")[-1].split(".")[0])
        )
        checkpoint_path = checkpoint_files[-1]
    assert os.path.exists(
        checkpoint_path
    ), f"Checkpoint file not found: {checkpoint_path}"
    checkpoint = torch.load(checkpoint_path)
    model.load_state_dict(checkpoint["model_state_dict"])
    optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
    return checkpoint["epoch"]

In [ ]:
def eval_on_test_set(
    model: torch.nn.Module,
    test_loader: torch.utils.data.DataLoader,
    device: str = "cpu",
) -> tuple[float, list[torch.Tensor]]:
    """
    Evaluate the model on the test set and compute the average Dice.

    Args:
        model (torch.nn.Module): The model to evaluate.
        test_loader (torch.utils.data.DataLoader): DataLoader for the test set.
        device (str): Device to run the evaluation on ("cpu" or "cuda").

    Returns:
        float: The average Dice over the test set.
        list[torch.Tensor]: List of predicted masks for the test set.
    """
    model.eval()
    dice_scores = []
    saved_masks = []
    with torch.no_grad():
        for imgs, masks in test_loader:
            imgs, masks = imgs.to(device), masks.to(device)
            pred_masks = model(imgs)
            pred_masks = torch.sigmoid(pred_masks)
            for pred_mask, true_mask in zip(pred_masks, masks):
                dice = compute_dice(pred_mask, true_mask)
                # Keep the saved masks on the CPU, so matplotlib can plot them
                # (and they don't fill up the GPU memory)
                saved_masks.append(pred_mask.cpu())
                dice_scores.append(dice)
    return np.mean(dice_scores), saved_masks

In [ ]:
# Set the device to GPU if available, otherwise use CPU
device = "cuda" if torch.cuda.is_available() else "cpu"

# Seed every random number generator, so repeated runs give the same results
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)  # also seeds the GPU
# Ask the GPU library (cuDNN) for repeatable algorithms instead of the fastest ones
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# Load the train, validation, and test datasets
data_dir_path = DATA_ROOT
data_map = load_dataset_map(data_dir_path)

# Augment only the training images: validation and test images must stay as they are
train_dataset = CustomDataset(data_map["train"], augment=True)
val_dataset = CustomDataset(data_map["val"])
test_dataset = CustomDataset(data_map["test"])


# Instantiate a pretrained U-Net model from the segmentation-models-pytorch library
model = smp.Unet(
    encoder_name="resnet18",
    encoder_weights="imagenet",
    in_channels=1,
    classes=1,
).to(device)

# Set the parameters map of the model: the encoder starts from pretrained weights,
# so it gets a lower learning rate (small updates keep what it has already learned),
# while the decoder and the segmentation head start from random weights and must
# learn from scratch, so they get a higher learning rate
param_groups = [
    {"params": model.encoder.parameters(), "lr": 1e-4},
    {"params": model.decoder.parameters(), "lr": 1e-3},
    {"params": model.segmentation_head.parameters(), "lr": 1e-3},
]

optimizer = torch.optim.AdamW(
    param_groups,
    lr=1e-3,
)
# Prepare the learning rate scheduler and optimizer
lr_scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    # For better performance, AdamW optimizer can be used instead of Adam
    # (AdamW is a variant of Adam that includes weight decay regularization)
    optimizer=optimizer,
    mode="min",
    factor=0.1,
    patience=5,
)


# Define the loss function for training the model
# Here we use the custom loss, but already implemented versions
# exist with many more configurations (see https://smp.readthedocs.io/en/latest/losses.html)
loss_fn = CustomDiceLoss()
# Set the maximum number of epochs for training the model
max_epochs = 50 if not RUN_SMALL_DATASET else 1
# Early stopping: stop training when the validation Dice has not improved
# for max_patience epochs in a row; a rise of the Dice smaller than
# min_delta is not counted as an improvement
max_patience = 10
min_delta = 1e-4

# Continue from the latest saved checkpoint only if asked to (see RESUME_TRAINING)
start_epoch = None
if RESUME_TRAINING:
    start_epoch = load_checkpoint(
        checkpoint_dir=str(MODEL_CHECKPOINT_FOLDER_PATH),
        model=model,
        optimizer=optimizer,
        epoch=None,
        is_best=False,
    )
if start_epoch is None:
    start_epoch = 0

In [ ]:
# Choose a proper batch size that can fit into the GPU memory (if available)
# or CPU memory
batch_size = 8
train_loader = torch.utils.data.DataLoader(
    train_dataset,
    shuffle=True,
    generator=torch.Generator().manual_seed(SEED),
    batch_size=batch_size,
    collate_fn=collate_fn,
)
val_loader = torch.utils.data.DataLoader(
    val_dataset,
    shuffle=False,
    batch_size=batch_size,
    collate_fn=collate_fn,
)
test_loader = torch.utils.data.DataLoader(
    test_dataset,
    shuffle=False,
    batch_size=batch_size,
    collate_fn=collate_fn,
)

# Print the initial performance of the model on the test set before training
initial_dice, _ = eval_on_test_set(model, test_loader, device=device)
print(f"Initial Dice on test set: {initial_dice:.4f}")

In [ ]:
# The seeded generator fixes the shuffling order and the random augmentations
# made inside the worker processes
train_loader = DataLoader(
    train_dataset,
    batch_size=8,
    shuffle=True,
    num_workers=4,
    generator=torch.Generator().manual_seed(SEED),
)
val_loader = DataLoader(val_dataset, batch_size=8, num_workers=4)

assert (
    start_epoch is not None and 0 <= start_epoch < max_epochs
), "Invalid start_epoch value"

device_type = torch.device(
    device
).type  # autocast needs "cuda"/"cpu", not a device object
# Mixed precision (float16) only on the GPU: on the CPU it is slow, and the sums
# inside the Dice loss overflow in float16 (loss stuck at 1.0, nothing is learned)
use_amp = device_type == "cuda"
scaler = torch.amp.GradScaler(device_type, enabled=use_amp)
best_val_dice = float("-inf")
epochs_without_improvement = 0

epoch_bar = tqdm(
    range(start_epoch, max_epochs), desc="Epochs", initial=start_epoch, total=max_epochs
)
for epoch in epoch_bar:
    # --- Training ---
    model.train()
    train_loss = 0.0
    train_bar = tqdm(train_loader, desc=f"Train {epoch + 1}", leave=False)
    for img, mask in train_bar:
        img, mask = img.to(device), mask.to(device)
        optimizer.zero_grad()
        with torch.autocast(
            device_type=device_type, dtype=torch.float16, enabled=use_amp
        ):
            # sigmoid turns the model's raw scores into 0-1 values,
            # the same way eval_on_test_set does before computing the Dice
            loss = loss_fn(torch.sigmoid(model(img)), mask)
        # Add scaling to avoid underflow in mixed precision training
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        train_loss += loss.item()
        train_bar.set_postfix(loss=f"{loss.item():.4f}")
    avg_train_loss = train_loss / len(train_loader)

    # --- Validation ---
    model.eval()
    val_loss = 0.0
    val_dice_scores = []
    with (
        torch.no_grad(),
        torch.autocast(device_type=device_type, dtype=torch.float16, enabled=use_amp),
    ):
        for img, mask in tqdm(val_loader, desc=f"Val {epoch + 1}", leave=False):
            img, mask = img.to(device), mask.to(device)
            pred_mask = torch.sigmoid(model(img))
            val_loss += loss_fn(pred_mask, mask).item()
            # Hard Dice of every image, the same metric used on the test set
            val_dice_scores += [compute_dice(p, m) for p, m in zip(pred_mask, mask)]
    avg_val_loss = val_loss / len(val_loader)
    avg_val_dice = float(np.mean(val_dice_scores))

    # --- Logging, checkpointing, LR schedule ---
    epoch_bar.set_postfix(
        train=f"{avg_train_loss:.4f}",
        val=f"{avg_val_loss:.4f}",
        val_dice=f"{avg_val_dice:.4f}",
        lr=f"{optimizer.param_groups[0]['lr']:.1e}",
    )

    # The best checkpoint is the one with the highest validation Dice, the metric
    # reported on the test set. The soft Dice loss also rewards confident probabilities,
    # so the epoch with the lowest loss is not always the one with the best 0/1 masks
    is_best = avg_val_dice > best_val_dice + min_delta
    if is_best:
        best_val_dice = avg_val_dice
        epochs_without_improvement = 0
    else:
        epochs_without_improvement += 1
    save_checkpoints(
        model=model,
        optimizer=optimizer,
        epoch=epoch + 1,
        checkpoint_dir=str(MODEL_CHECKPOINT_FOLDER_PATH),
        is_best=is_best,
    )
    # The learning rate schedule follows the validation loss, which changes more
    # smoothly from one epoch to the next than the hard Dice
    lr_scheduler.step(avg_val_loss)

    if epochs_without_improvement >= max_patience:
        tqdm.write(
            f"Early stopping after epoch {epoch + 1}: "
            f"no validation Dice improvement for {max_patience} epochs"
        )
        break

In [ ]:
# Evaluate the model on the test set after training, using the best checkpoint
# (highest validation Dice) instead of the weights of the last epoch
best_epoch = load_checkpoint(
    checkpoint_dir=str(MODEL_CHECKPOINT_FOLDER_PATH),
    model=model,
    optimizer=optimizer,
    is_best=True,
)
print(f"Loaded the best checkpoint (epoch {best_epoch})")
test_dice, test_pred_masks = eval_on_test_set(model, test_loader, device=device)
print("Final Dice on test set: {:.4f}".format(test_dice))

In [ ]:
def visualize_samples(
    test_dataset: CustomDataset,
    saved_predicted_masks: list[torch.Tensor],
    selected_indices: list[int],
):
    """
    Visualize some examples of the test set with their corresponding ground truth masks
    and predicted masks.

    Args:
        test_dataset (CustomDataset): The test dataset.
        saved_predicted_masks (list[torch.Tensor]): List of predicted masks for the test set.
        selected_indices (list[int]): List of indices of the samples to visualize.
    """
    for i in selected_indices:
        img = test_dataset[i][0]
        mask = test_dataset[i][1]
        pred_mask = saved_predicted_masks[i]

        # Visualize the image, ground truth mask, and predicted mask
        plt.subplot(1, 3, 1)
        plt.imshow(img.squeeze(), cmap="gray")
        plt.title("Image")
        plt.axis("off")

        plt.subplot(1, 3, 2)
        plt.imshow(mask.squeeze(), cmap="gray")
        plt.title("Ground Truth")
        plt.axis("off")

        plt.subplot(1, 3, 3)
        plt.imshow(pred_mask.squeeze(), cmap="gray")
        plt.title("Predicted")
        plt.axis("off")

        plt.show()

In [ ]:
# Visualize some examples of the test set with their corresponding ground truth masks
# and predicted masks
selected_indices = [0, 1, 2]

visualize_samples(
    test_dataset=test_dataset,
    saved_predicted_masks=test_pred_masks,
    selected_indices=selected_indices,
)